In [39]:
from reasoning_from_scratch.qwen3 import download_qwen3_small,Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import Qwen3Model,QWEN_CONFIG_06_B,KVCache
from pathlib import Path
import torch.nn as nn
import torch

In [4]:
download_qwen3_small(kind="base",tokenizer_only=False,out_dir="../qwen")

✓ ..\qwen\qwen3-0.6B-base.pth already up-to-date


In [5]:
tokenizer_path=Path("..")/"qwen"/"tokenizer-base.json"
tokenizer=Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [6]:
device=torch.device("cpu")

In [7]:
model_path = Path("..") / "qwen" / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [8]:
input_ids=torch.tensor(tokenizer.encode("Hey who are you"))
input_ids=input_ids.unsqueeze(0).to(device)

In [41]:

def text_gen(prompt: str, model, n_tokens, eos_token_id=151643):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()
    cache=KVCache(n_layers=model.cfg["n_layers"])

    for _ in range(n_tokens):
        output = model(input_ids)
        output = output.squeeze(0)
        last_logit = output[-1, :]
        tokenid = torch.argmax(last_logit, dim=-1, keepdim=True)

        if tokenid.item() == eos_token_id:
            break

        print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
        yield tokenid

        # outputids=torch.concat((outputids,tokenid),dim=-1)
        input_ids = torch.concat(
            (input_ids, tokenid.unsqueeze(0)), dim=-1
        )

    # return tokenizer.decode(list(outputids))


In [42]:

def text_gen(prompt: str,model,n_tokens,eos_token_id=151643,temperature=1.0,top_k=50):

    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()
    cache=KVCache(n_layers=model.cfg["n_layers"])

    for _ in range(n_tokens):
        output = model(input_ids)
        output = output.squeeze(0)
        last_logit = output[-1, :]

        #temp scaling
        last_logit = last_logit / temperature

        # topk sampling
        topk_logits, topk_indices = torch.topk(last_logit, top_k)
        probs = torch.softmax(topk_logits, dim=-1)
        sampled_index = torch.multinomial(probs, num_samples=1)
        tokenid = topk_indices[sampled_index].reshape(1)

        if tokenid.item() == eos_token_id:
            break

        print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
        yield tokenid

        # outputids=torch.concat((outputids,tokenid),dim=-1)
        input_ids = torch.concat(
            (input_ids, tokenid.unsqueeze(0)), dim=-1
        )

    # return tokenizer.decode(list(outputids))


In [44]:
for token in text_gen("What is 2+2", model,temperature=1,n_tokens=30):
    pass

, and how do you solve 5+2? What are the steps for performing mental math addition, and how do you practice these skills?

2